# Anime FFmpeg Re-Encoder Notebook
Optimized for anime episode-by-episode encoding (Support HEVC 10-bit NVENC/x265)

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

!apt update -qq && apt install -y ffmpeg aria2 mkvtoolnix mediainfo -qq
!pip install tqdm -q

import os
from pathlib import Path
import glob
import subprocess
from tqdm.notebook import tqdm
from IPython.display import clear_output

clear_output()
print("Setup selesai!")
print("FFmpeg version:")
!ffmpeg -version | head -n 1
print("Aria2 version:")
!aria2c --version | head -n 1

In [ ]:

TEMP_DIR = "/content/temp_downloads"
DRIVE_OUTPUT_FOLDER = "/content/drive/MyDrive/Anime/Downloads"
os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_FOLDER, exist_ok=True)

USE_GPU = True  # True = NVENC GPU, False = x265 CPU

PROFILES = {
    "Balanced 1080p": {"height": 1080, "crf_qp": 24, "audio": "libopus", "audio_bitrate": "128k"},
    "High Quality 1080p": {"height": 1080, "crf_qp": 22, "audio": "libopus", "audio_bitrate": "160k"},
    "Small 720p": {"height": 720, "crf_qp": 26, "audio": "libopus", "audio_bitrate": "112k"},
    "Mobile 480p": {"height": 480, "crf_qp": 28, "audio": "libopus", "audio_bitrate": "96k"},
    "Mobile 360p": {"height": 360, "crf_qp": 30, "audio": "libopus", "audio_bitrate": "96k"},
}

SELECTED_PROFILE = "Mobile 360p"
profile = PROFILES[SELECTED_PROFILE]

DOWNLOAD_URL = "https://syqrel.si/download/4254223.torrent"  # URL torrent/magnet/file
ARIA2_CONCURRENT = 16  # Number of download connections
ARIA2_SPLIT = 16  # Split connections
ARIA2_MAX_JOBS = 5  # Max parallel downloads
ARIA2_SEED_TIME = 0  # Seed time in seconds (0 = no seeding)
ARIA2_BT_STOP_TIMEOUT = 300  # BitTorrent stop timeout

ENABLE_DOWNLOAD = True  # Enable download step
ENABLE_ENCODE = True  # Enable encoding step
ENABLE_MOVE_TO_DRIVE = True  # Enable move to Drive step
ENABLE_AUTO_SHUTDOWN = True  # Enable auto shutdown after completion
SHUTDOWN_DELAY = 10  # Delay before shutdown (seconds)

VIDEO_EXTENSIONS = ["**/*.mkv", "**/*.mp4", "**/*.webm"]  # Video file patterns to search

print("=" * 60)
print("CONFIGURATION LOADED")
print("=" * 60)
print(f"Profile: {SELECTED_PROFILE}")
print(f"â†’ {profile['height']}p | CRF/QP: {profile['crf_qp']} | Audio: {profile['audio']}")
print(f"â†’ Mode: {'NVENC GPU' if USE_GPU else 'x265 CPU'}")
print(f"â†’ Temp folder: {TEMP_DIR}")
print(f"â†’ Drive output: {DRIVE_OUTPUT_FOLDER}")
print(f"â†’ Download URL: {DOWNLOAD_URL}")
print(f"â†’ Enable download: {ENABLE_DOWNLOAD}")
print(f"â†’ Enable encode: {ENABLE_ENCODE}")
print(f"â†’ Enable move to Drive: {ENABLE_MOVE_TO_DRIVE}")
print(f"â†’ Enable auto shutdown: {ENABLE_AUTO_SHUTDOWN}")
print("=" * 60)

In [ ]:

def download_file(url: str) -> bool:
    """Download file/torrent/magnet using aria2"""
    print(f"\n{'='*60}")
    print("DOWNLOADING")
    print(f"{'='*60}")
    print(f"URL: {url}")

    cmd = f"""aria2c \
  -x {ARIA2_CONCURRENT} -s {ARIA2_SPLIT} -j {ARIA2_MAX_JOBS} \
  --seed-time={ARIA2_SEED_TIME} \
  --bt-stop-timeout={ARIA2_BT_STOP_TIMEOUT} \
  --summary-interval=5 \
  --dir="{TEMP_DIR}" \
  --allow-overwrite=true \
  --auto-file-renaming=true \
  --file-allocation=falloc \
  "{url}"
"""
    try:
        result = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
        print("\nDownload complete. Files in temp folder:")
        subprocess.run(["ls", "-lh", TEMP_DIR])
        subprocess.run(["du", "-sh", TEMP_DIR])
        return True
    except subprocess.CalledProcessError as e:
        print(f"Download error: {e}")
        if e.stderr:
            print(e.stderr)
        return False


def encode_video(input_path: str, profile: dict, use_gpu: bool = True):
    """Encode video file using FFmpeg"""
    input_path = Path(input_path)
    if not input_path.is_file():
        print(f"File not found: {input_path}")
        return None

    base = input_path.stem
    suffix = f"_{profile['height']}p"
    output_path = Path(TEMP_DIR) / f"{base}{suffix}.mkv"

    height = profile["height"]
    crf = profile["crf_qp"]
    audio_codec = profile["audio"]
    audio_br = profile.get("audio_bitrate", "128k")

    if audio_codec == "copy":
        audio_args = ["-c:a", "copy"]
    else:
        audio_args = ["-c:a", audio_codec, "-b:a", audio_br, "-ac", "2", "-vbr", "on"]

    if use_gpu:
        hw = ["-init_hw_device", "cuda=cuda", "-filter_hw_device", "cuda"]
        vcodec = "hevc_nvenc"
        preset = "p7"
        tune = "hq"
        vf = f"format=yuv420p10le,hwupload_cuda,scale_cuda=-2:{height}:interp_algo=lanczos:force_original_aspect_ratio=decrease"
        v_args = ["-c:v", vcodec, "-preset", preset, "-tune", tune,
                  "-cq", str(crf), "-qmin", str(crf), "-qmax", str(crf+4),
                  "-rc", "vbr", "-rc-lookahead", "48", "-b:v", "0"]
    else:
        hw = []
        vcodec = "libx265"
        preset = "slow"
        vf = f"scale=-2:{height}:flags=lanczos+accurate_rnd+full_chroma_int+full_chroma_inp,format=yuv420p10le"
        x265_params = "aq-mode=3:psy-rd=2.0:psy-rdoq=2.0:no-sao=1:deblock=-1,-1"
        v_args = ["-c:v", vcodec, "-preset", preset, "-crf", str(crf), "-x265-params", x265_params]

    vf += ",setparams=color_primaries=bt709:color_trc=bt709:colorspace=bt709"

    cmd = [
        "ffmpeg", "-y", "-vsync", "0",
        *hw,
        "-i", str(input_path),
        "-map", "0", "-map", "-0:d?",
        "-vf", vf,
        *v_args,
        *audio_args,
        "-c:s", "copy",
        "-metadata:s:v:0", f"title=Encoded - {SELECTED_PROFILE}",
        str(output_path)
    ]

    print(f"\n{'='*60}")
    print("ENCODING")
    print(f"{'='*60}")
    print(f"Input: {input_path.name}")
    print(f"Output: {output_path.name}")

    try:
        subprocess.run(cmd, check=True)
        print(f"Encoding complete: {output_path}")
        subprocess.run(["ls", "-lh", str(output_path)])
        return str(output_path)
    except Exception as e:
        print(f"Encoding error: {e}")
        return None


def find_latest_video() -> str:
    """Find latest video file in temp directory"""
    downloaded_files = []
    for ext in VIDEO_EXTENSIONS:
        downloaded_files.extend(glob.glob(os.path.join(TEMP_DIR, ext), recursive=True))

    if not downloaded_files:
        print("No video files found in temp folder")
        print("Current folder contents:")
        subprocess.run(["ls", "-R", "-lh", TEMP_DIR], check=False)
        return None

    latest_input = max(downloaded_files, key=os.path.getmtime)
    print(f"Latest file found: {latest_input}")
    print(f"Filename: {Path(latest_input).name}")
    return latest_input


def move_to_drive(encoded_path: str) -> bool:
    """Move encoded file to Google Drive"""
    if not encoded_path:
        return False

    final_name = Path(encoded_path).name
    drive_final_path = os.path.join(DRIVE_OUTPUT_FOLDER, final_name)

    print(f"\n{'='*60}")
    print("MOVING TO DRIVE")
    print(f"{'='*60}")
    print(f"Source: {encoded_path}")
    print(f"Destination: {drive_final_path}")

    try:
        import shutil
        shutil.move(encoded_path, drive_final_path)
        subprocess.run(["ls", "-lh", drive_final_path], check=False)
        print(f"Successfully moved to Google Drive")
        return True
    except Exception as e:
        print(f"Move error: {e}")
        return False


def check_and_shutdown(encoded_filename: str) -> None:
    """Check if file exists in Drive and shutdown if enabled"""
    if not ENABLE_AUTO_SHUTDOWN:
        print("Auto shutdown disabled")
        return

    expected_suffix = f"_{profile['height']}p.mkv"
    drive_final_path = os.path.join(DRIVE_OUTPUT_FOLDER, encoded_filename)

    print(f"\n{'='*60}")
    print("SHUTDOWN CHECK")
    print(f"{'='*60}")

    if Path(drive_final_path).is_file():
        file_size = Path(drive_final_path).stat().st_size / (1024**3)
        print(f"SUCCESS: File exists in Google Drive")
        print(f"Path: {drive_final_path}")
        print(f"Size: {file_size:.2f} GB")
        print(f"\nAll processes completed. Shutting down runtime in {SHUTDOWN_DELAY} seconds...")

        import time
        time.sleep(SHUTDOWN_DELAY)

        from google.colab import runtime
        runtime.unassign()
    else:
        print(f"File not found in Drive: {drive_final_path}")
        print("â†’ Shutdown cancelled. Check if move operation succeeded.")

In [ ]:

encoded_filename = None

if ENABLE_DOWNLOAD:
    download_success = download_file(DOWNLOAD_URL)
    if not download_success:
        print("Download failed. Stopping workflow.")
else:
    print("Download step disabled")
    download_success = True

if ENABLE_ENCODE and download_success:
    latest_video = find_latest_video()
    if latest_video:
        encoded_path = encode_video(latest_video, profile, use_gpu=USE_GPU)
        if encoded_path:
            encoded_filename = Path(encoded_path).name
    else:
        print("No video file found to encode")
else:
    print("Encode step disabled or download failed")

if ENABLE_MOVE_TO_DRIVE and encoded_filename:
    encoded_path = os.path.join(TEMP_DIR, encoded_filename)
    move_success = move_to_drive(encoded_path)

    if move_success:
        check_and_shutdown(encoded_filename)
else:
    print("Move to Drive step disabled or no encoded file")